In [1]:
import os
import pandas as pd
from neo4j import GraphDatabase,basic_auth
from utils.Evaluation.output_metrics import *
from utils.Evaluation.cypher_metrics import *
from CyVer import SyntaxValidator, SchemaValidator, PropertiesValidator


## Notebook for:
*  overview of all results 
*  Evaluate with CyVer if the Original Queries of the datasets are KG Valid & update in all the output results files
*  Set  Pass=None and Output Jaccard = None for all not KG valid queries  that could not have been executed in the database

In [2]:
def process_xlsx_in_subfolders(root_folder):
    results = []

    for subdir, dirs, files in os.walk(root_folder):
        subfolder_name = os.path.basename(subdir)
        if subfolder_name == 'old':
            continue

        for file in files:
            if file.endswith('.xlsx'):
                file_path = os.path.join(subdir, file)
                try:
                    df = pd.read_excel(file_path)
                    has_column = 'Original Schema Valid' in df.columns
                    results.append({
                        'model': subfolder_name,
                        'num_rows': len(df),
                        'evaluate_original': has_column
                    })
                except Exception as e:
                    print(f"Failed to read {file_path}: {e}")
    df_results = pd.DataFrame(results)
    return  df_results.sort_values(by='num_rows', ascending=False).reset_index(drop=True)

# Example usage
workflow = 'extended_workflow/Outputs_Evaluation/'
dataset = 'BenchmarkDataset2025'
root_folder = os.path.join(workflow, dataset)

print("Processing folder:", root_folder)
df_results_extended_bench = process_xlsx_in_subfolders(root_folder)
print(df_results_extended_bench)

workflow = 'basic_workflow/Outputs_Evaluation/'
dataset = 'BenchmarkDataset2025'
root_folder = os.path.join(workflow, dataset)
print("Processing folder:", root_folder)
df_results_basic_bench = process_xlsx_in_subfolders(root_folder)
print(df_results_basic_bench)

Processing folder: extended_workflow/Outputs_Evaluation/BenchmarkDataset2025
                           model  num_rows  evaluate_original
0                    llama3_2_3b      2128               True
1  finetuned_azzedde_llama3_1_8b      2128               True
2                         gpt_4o      2128               True
3                    llama3_1_8b      2128               True
4                      gemma2_9b      2017               True
5      finetuned_gemma2_9b_local      2012               True
Processing folder: basic_workflow/Outputs_Evaluation/BenchmarkDataset2025
                           model  num_rows  evaluate_original
0              llama3_2_3b_local      2128               True
1  finetuned_azzedde_llama3_1_8b      2128               True
2                    llama3_1_8b      2128               True
3                         gpt_4o      2128               True
4                      gemma2_9b      2012               True
5      finetuned_gemma2_9b_local      2011 

In finetuned_gemma2_9b_local:
 - Extended:
    - Skipping  database_name == 'offshoreleaks' to avoid memory error / context length (115 rows --> 2128-115 =2013)
    - Skipping database_name == 'openstreetmap' to avoid memory error / context length (2013-1 = 2012) correct
- Basic:
  - Skipping  database_name == 'offshoreleaks' to avoid memory error / context length (115 rows --> 2128-115 =2013)
  - Skipping database_name == 'openstreetmap' to avoid memory error / context length (2013-1 = 2012) 
  - In one question it generated no quert --> 2012-1 = 2011 correct

In gemma2_9b:
- Extended:
    - Skipping  database_name == 'offshoreleaks' to avoid memory error / context length (115 rows --> 2128-115 =2013)
    - Skipping  database_name == 'openstreetmap' to avoid memory error / context length (2013-1 = 2012)
    BUT SOME of offshoreleaks were run (2017) correct
- Basic:
   - Skipping  database_name == 'offshoreleaks' to avoid memory error / context length (115 rows --> 2128-115 =2013)
    - Skipping  database_name == 'openstreetmap' to avoid memory error / context length (2013-1 = 2012)
                


## Evaluate with CyVer the Original queries of the Neo4j Benchmark dataset

In [17]:
df_with_original_valid = pd.read_excel('extended_workflow/Outputs_Evaluation/BenchmarkDataset2025/gemma2_9b/gemma2_9b.xlsx')
# df_with_original_valid['Original Properties Valid'].unique()
filtered_df = df_with_original_valid[
    (df_with_original_valid['Original Schema Valid'] == 1) &
    ((df_with_original_valid['Original Properties Valid'] == 1) | (df_with_original_valid['Original Properties Valid'].isna())) &
    (df_with_original_valid['Original Syntax Valid'] == True)
]
filtered_df

,question,schema,cypher,data_source,instance_id,database_reference_alias,url,database_name,username,password,...,Properties Valid,Properties Metadata,Pass@1 Output,Jaccard Output,BLEU,Rouge-L Precision,Rouge-L Recall,Rouge-L F1-score,Jaro Similarity,Jaccard Similarity
0,Identify the 5 suppliers with the highest aver...,Node properties:\n- **Product**\n - `productN...,MATCH (s:Supplier)-[:SUPPLIES]->(p:Product)\nW...,neo4jLabs_synthetic_gpt4o,instance_id_44523,neo4jlabs_demo_db_northwind,neo4j+s://demo.neo4jlabs.com,northwind,northwind,northwind,...,1.0,[],1.0,1.000000,0.558353,0.83,0.70,0.76,0.903784,0.521739
1,List the first 3 genres of movies with the low...,Node properties:\n- **Movie**\n - `average_vo...,MATCH (m:Movie)-[:IN_GENRE]->(g:Genre)\nWITH g...,neo4jLabs_synthetic_gpt4o,instance_id_44513,neo4jlabs_demo_db_neoflix,neo4j+s://demo.neo4jlabs.com,neoflix,neoflix,neoflix,...,1.0,[],0.0,0.000000,0.319772,0.55,0.38,0.45,0.866389,0.391304
2,What is the total number of users who are VIPs...,Node properties:\n- **Stream**\n - `createdAt...,MATCH (user:User)-[:VIP]->(stream:Stream)\nWIT...,neo4jLabs_synthetic_gemini,instance_id_33826,neo4jlabs_demo_db_twitch,neo4j+s://demo.neo4jlabs.com,twitch,twitch,twitch,...,NaN,[],NaN,NaN,0.310457,0.42,0.44,0.43,0.827150,0.346154
3,"""show me the oldest tweet id""","Node properties:\nUser {betweenness: FLOAT, lo...",MATCH (t:Tweet)\nRETURN t.id\nORDER BY t.creat...,neo4j_crowdsourced,instance_id_27353,neo4jlabs_demo_db_twitter,neo4j+s://demo.neo4jlabs.com,twitter,twitter,twitter,...,1.0,[],1.0,1.000000,0.470871,0.77,0.71,0.74,0.925391,0.461538
4,List the 5 most recent users who started follo...,Node properties:\n- **User**\n - `betweenness...,MATCH (neo4j:Me {screen_name: 'neo4j'})<-[:FOL...,neo4jLabs_synthetic_gpt4o,instance_id_42820,neo4jlabs_demo_db_twitter,neo4j+s://demo.neo4jlabs.com,twitter,twitter,twitter,...,1.0,[],0.0,0.000000,0.193103,0.55,0.33,0.41,0.666214,0.269231
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2011,Find the top 3 questions with the most comments.,Node properties:\n- **Question**\n - `link`: ...,MATCH (q:Question)<-[:COMMENTED_ON]-(c:Comment...,neo4jLabs_synthetic_gpt4o,instance_id_42345,neo4jlabs_demo_db_stackoverflow2,neo4j+s://demo.neo4jlabs.com,stackoverflow2,stackoverflow2,stackoverflow2,...,1.0,[],0.0,0.666667,0.602616,0.87,0.59,0.70,0.924701,0.761905
2012,"Who are the actors in ""Avatar""? I want to use ...",Node properties are the following:\nMovie {pos...,MATCH (m:Movie {title:'Avatar'})\nCALL (m) {\n...,20250307_datacleanup,instance_id_45625,neo4jlabs_demo_db_recommendations,neo4j+s://demo.neo4jlabs.com,recommendations,recommendations,recommendations,...,1.0,[],0.0,0.000000,0.451337,1.00,0.59,0.74,0.859707,0.555556
2013,How to create a new Movie with a Genre? I want...,Node properties are the following:\nMovie {pos...,"CREATE (m:Movie {\n title: ""Matrix Reborn"",\n...",20250307_datacleanup,instance_id_45626,neo4jlabs_demo_db_recommendations,neo4j+s://demo.neo4jlabs.com,recommendations,recommendations,recommendations,...,1.0,[],0.0,0.000000,0.417578,0.35,0.33,0.34,0.736059,0.318182
2014,How to remove all Movies older than the year 2...,Node properties are the following:\nMovie {pos...,MATCH (m:Movie)\nWHERE m.year < 2000\nCALL (m)...,20250307_datacleanup,instance_id_45627,neo4jlabs_demo_db_recommendations,neo4j+s://demo.neo4jlabs.com,recommendations,recommendations,recommendations,...,1.0,[],0.0,0.000000,0.468314,1.00,0.64,0.78,0.897636,0.636364


In [24]:
print (f'{len(df_with_original_valid) - len(filtered_df)} original queries are not KG valid')

48 original queries are not KG valid


## Evaluate offshoreleaks and openstreetmap original queries in gpt4o

In [31]:
def postprocess_output_cypher_by_neo4j(output_cypher: str) -> str:
    # Remove any explanation. E.g.  MATCH...\n\n**Explanation:**\n\n -> MATCH...
    # Remove cypher indicator. E.g.```cypher\nMATCH...```` --> MATCH...
    # Note: Possible to have both:
    #   E.g. ```cypher\nMATCH...````\n\n**Explanation:**\n\n --> MATCH...
    
    # Handle escaped newline characters (\\n -> real \n)
    output_cypher = output_cypher.encode().decode('unicode_escape') # decode unicode escape sequences

    partition_by = "**Explanation:**"
    output_cypher, _, _ = output_cypher.partition(partition_by)
    output_cypher = output_cypher.strip("`\n")
    output_cypher = output_cypher.lstrip("cypher\n")
    output_cypher = output_cypher.strip("`\n ")
    output_cypher = output_cypher.replace('\n', ' ') ## added to fix benchmark dataset

    return output_cypher


In [35]:

df_gpt4o = pd.read_excel('extended_workflow/Outputs_Evaluation/BenchmarkDataset2025/gpt_4o/gpt_4o.xlsx')
# initialize columns
df_gpt4o['Original Syntax Valid'] = None
df_gpt4o['Original Syntax Metadata'] = None
df_gpt4o['Original Schema Valid'] = None
df_gpt4o['Original Schema Metadata'] = None
df_gpt4o['Original Properties Valid'] = None
df_gpt4o['Original Properties Metadata'] = None
for i, row in df_gpt4o.iterrows():
    database_url = row['url']
    database_name = row['database_name']
    database_user = row['username']
    database_password = row['password']

    if database_name == 'offshoreleaks' or database_name =='openstreetmap':
        print(i, database_name)
        if pd.isna(row['Original Syntax Valid']):
            driver = GraphDatabase.driver(database_url, auth=basic_auth(database_user, database_password) )
            
            syntax_validator = SyntaxValidator(driver)
            schema_validator = SchemaValidator(driver)
            properties_validator = PropertiesValidator(driver)

            original_query = row['cypher']
            original_query = postprocess_output_cypher_by_neo4j(original_query) #apply their prostprocess function to make sure the original queries are valid
            ## CyVer validation for original query
            og_syntax_correct, og_syntax_metadata = syntax_validator.validate(original_query, database_name=database_name)
            og_schema_valid, og_schema_metadata = schema_validator.validate(original_query, database_name=database_name)
            og_properties_valid, og_properties_metadata = properties_validator.validate(original_query,database_name=database_name, strict=False)
            
            print(f"Original Syntax Valid: {og_syntax_correct}")
            print(f"Original Schema Valid: {og_schema_valid}")
            print(f"Original Properties Valid: {og_properties_valid}")

            df_gpt4o.at[i, 'Original Syntax Valid'] = og_syntax_correct        
            df_gpt4o.at[i, 'Original Syntax Metadata'] = str(og_syntax_metadata)
            df_gpt4o.at[i, 'Original Schema Valid'] = og_schema_valid
            df_gpt4o.at[i, 'Original Schema Metadata'] = str(og_schema_metadata)
            df_gpt4o.at[i, 'Original Properties Valid'] = og_properties_valid
            df_gpt4o.at[i, 'Original Properties Metadata'] = str(og_properties_metadata)
            df_gpt4o.to_pickle('extended_workflow/Outputs_Evaluation/BenchmarkDataset2025/gpt_4o/gpt_4o.pkl')
            df_gpt4o.to_excel('extended_workflow/Outputs_Evaluation/BenchmarkDataset2025/gpt_4o/gpt_4o.xlsx', index=False)
        else:
            print('Calculated')



10 offshoreleaks
Original Syntax Valid: True
Original Schema Valid: 1.0
Original Properties Valid: 1.0
13 offshoreleaks
Original Syntax Valid: True
Original Schema Valid: 1.0
Original Properties Valid: 1.0
38 offshoreleaks
Original Syntax Valid: True
Original Schema Valid: 1.0
Original Properties Valid: 1.0
43 offshoreleaks
Original Syntax Valid: True
Original Schema Valid: 1.0
Original Properties Valid: 1.0
87 offshoreleaks
Original Syntax Valid: True
Original Schema Valid: 1.0
Original Properties Valid: 1.0
125 offshoreleaks
Original Syntax Valid: True
Original Schema Valid: 1.0
Original Properties Valid: 1.0
159 offshoreleaks
Original Syntax Valid: True
Original Schema Valid: 1.0
Original Properties Valid: 1.0
185 offshoreleaks
Original Syntax Valid: True
Original Schema Valid: 1.0
Original Properties Valid: 1.0
195 offshoreleaks
Original Syntax Valid: True
Original Schema Valid: 1.0
Original Properties Valid: 1.0
196 offshoreleaks
Original Syntax Valid: True
Original Schema Valid: 

In [ ]:
df_gemma = pd.read_excel('extended_workflow/Outputs_Evaluation/BenchmarkDataset2025/gemma2_9b/gemma2_9b.xlsx')

key_columns = ['question',	'cypher', 'database_name']
# Columns you want to fill
cols_to_fill = [
    'Original Syntax Valid',
    'Original Syntax Metadata',
    'Original Schema Valid',
    'Original Schema Metadata',
    'Original Properties Valid',
    'Original Properties Metadata'
]

# Merge df_gpt4o with df_gemma on keys, adding suffix _gemma to the filler columns
merged = df_gpt4o.merge(
    df_gemma,
    on=key_columns,
    how='left',
    suffixes=('', '_gemma')
)
# Fill missing values in df_gpt4o columns with values from df_gemma (merged columns)
for col in cols_to_fill:
    merged[col] = merged[col].combine_first(merged[f"{col}_gemma"])

# Drop the extra gemma columns
merged = merged.drop(columns=[col for col in merged.columns if col.endswith('_gemma')])

# Now merged has df_gpt4o rows with missing values filled from df_gemma
df_filled = merged


In [48]:
df_filled.to_pickle('extended_workflow/Outputs_Evaluation/BenchmarkDataset2025/gpt_4o/gpt_4o.pkl')
df_filled.to_excel('extended_workflow/Outputs_Evaluation/BenchmarkDataset2025/gpt_4o/gpt_4o.xlsx', index=False)

## How many original queries are not KG valid

In [ ]:
df_with_original_valid = pd.read_excel('extended_workflow/Outputs_Evaluation/BenchmarkDataset2025/gpt_4o/gpt_4o.xlsx')
# df_with_original_valid['Original Properties Valid'].unique()
kg_valid = (
    (df_with_original_valid['Original Schema Valid'] == 1) &
    ((df_with_original_valid['Original Properties Valid'] == 1) | (df_with_original_valid['Original Properties Valid'].isna())) &
    (df_with_original_valid['Original Syntax Valid'] == True)
)
invalid_df = df_with_original_valid[~kg_valid]
valid_df =  df_with_original_valid[kg_valid]
# Add the boolean result as a new column
df_with_original_valid['Original_KG_Valid'] = kg_valid

print (f'{len(invalid_df) } original queries are not KG valid')
df_with_original_valid.to_pickle('extended_workflow/Outputs_Evaluation/BenchmarkDataset2025/gpt_4o/gpt_4o.pkl')
df_with_original_valid.to_excel('extended_workflow/Outputs_Evaluation/BenchmarkDataset2025/gpt_4o/gpt_4o.xlsx', index=False)


53 original queries are not KG valid


In [54]:
invalid_df[['cypher', 'database_name', 'Original Syntax Valid',
    'Original Syntax Metadata',
    'Original Schema Valid',
    'Original Schema Metadata',
    'Original Properties Valid',
    'Original Properties Metadata']]

,cypher,database_name,Original Syntax Valid,Original Syntax Metadata,Original Schema Valid,Original Schema Metadata,Original Properties Valid,Original Properties Metadata
26,MATCH (o:Order {shipCountry: 'Denmark'})-[:ORD...,northwind,True,[],1.000000,[],0.500000,[{'code': 'InvalidPropertyAccessByLabelWarning...
30,"MATCH (e1:Entity {name: ""Deutsche Bank AG""})\n...",fincen,True,[],0.888889,"[{'code': 'WrongDirectionPathWarning', 'descri...",1.000000,[]
90,MATCH (b:Business {city: 'Burlingame'})-[:REVI...,grandstack,True,[],0.666667,"[{'code': 'WrongDirectionPathWarning', 'descri...",1.000000,[]
99,MATCH (c:Country)<-[:COUNTRY]-(e:Entity)-[:ORI...,fincen,True,[],0.833333,"[{'code': 'WrongDirectionPathWarning', 'descri...",1.000000,[]
160,MATCH (p:Product)-[o:ORDERS]->(:Order)\nWITH p...,northwind,True,[],0.666667,"[{'code': 'WrongDirectionPathWarning', 'descri...",1.000000,[]
283,MATCH (a:Address)\nWHERE a.jurisdiction_descri...,offshoreleaks,True,[],1.000000,[],0.000000,[{'code': 'InvalidPropertyAccessByLabelWarning...
284,MATCH (m:Movie)-[r:RATED]->()\nWHERE m.year >=...,recommendations,True,[],0.666667,"[{'code': 'WrongDirectionPathWarning', 'descri...",1.000000,[]
302,MATCH (e:Entity)-[:ORIGINATOR]->(f:Filing)\nWI...,fincen,True,[],0.833333,"[{'code': 'WrongDirectionPathWarning', 'descri...",1.000000,[]
359,MATCH (p:Product {productName: 'Aniseed Syrup'...,northwind,True,[],0.666667,"[{'code': 'WrongDirectionPathWarning', 'descri...",0.333333,[{'code': 'InvalidPropertyAccessByLabelWarning...
366,MATCH (prodCompany:ProductionCompany {name: 'P...,neoflix,True,[],0.666667,"[{'code': 'WrongDirectionPathWarning', 'descri...",1.000000,[]


Add Original KG Valid column in gemma

In [71]:
df_with_original_valid = pd.read_excel('extended_workflow/Outputs_Evaluation/BenchmarkDataset2025/gemma2_9b/gemma2_9b.xlsx')
# df_with_original_valid['Original Properties Valid'].unique()
kg_valid = (
    (df_with_original_valid['Original Schema Valid'] == 1) &
    ((df_with_original_valid['Original Properties Valid'] == 1) | (df_with_original_valid['Original Properties Valid'].isna())) &
    (df_with_original_valid['Original Syntax Valid'] == True)
)
invalid_df = df_with_original_valid[~kg_valid]
valid_df =  df_with_original_valid[kg_valid]
# Add the boolean result as a new column
df_with_original_valid['Original_KG_Valid'] = kg_valid

print (f'{len(invalid_df) } original queries are not KG valid')
df_with_original_valid.to_pickle('extended_workflow/Outputs_Evaluation/BenchmarkDataset2025/gemma2_9b/gemma2_9b.pkl')
df_with_original_valid.to_excel('extended_workflow/Outputs_Evaluation/BenchmarkDataset2025/gemma2_9b/gemma2_9b.xlsx', index=False)


48 original queries are not KG valid


# Add to all the models' results the original queries evaluation

In [19]:
def adding_kg_valid(df_new):
    df_gpt4o = pd.read_excel('extended_workflow/Outputs_Evaluation/BenchmarkDataset2025/gpt_4o/gpt_4o.xlsx')
    key_columns = ['question',	'cypher', 'database_name']
    # Columns you want to fill
    cols_to_fill = [
    'Original Syntax Valid',
    'Original Syntax Metadata',
    'Original Schema Valid',
    'Original Schema Metadata',
    'Original Properties Valid',
    'Original Properties Metadata',
    'Original_KG_Valid'
    ]
    print(len(df_new))
    df_merged = df_new.merge(
        df_gpt4o[key_columns + cols_to_fill],
        on=key_columns,
        how='left')
    
    return df_merged

In [30]:
root_folder = 'basic_workflow/Outputs_Evaluation/BenchmarkDataset2025' #
# df_gpt4o = pd.read_excel('extended_workflow/Outputs_Evaluation/BenchmarkDataset2025/gpt_4o/gpt_4o.xlsx')

for subdir, dirs, files in os.walk(root_folder):
    subfolder_name = os.path.basename(subdir)
    if subfolder_name == 'old':
        continue
    for file in files:
        if file.endswith('.xlsx'):
            file_path = os.path.join(subdir, file)
            try:
                if not file == 'gpt_4o.xlsx' :
                    continue
                print(file)
                df = pd.read_excel(file_path)
                print('Merging ', file )
                df_new = adding_kg_valid(df)
                # Save to Excel (overwrite the same file)
                df_new.to_excel(file_path, index=False)

                # Save to Pickle
                pickle_file_path = file_path.replace('.xlsx', '.pkl')
                df_new.to_pickle(pickle_file_path)
                print('saved')
            except Exception as e:
                print(f"Failed to read {file_path}: {e}")



gpt_4o.xlsx
Merging  gpt_4o.xlsx
2128
saved


In [31]:
root_new = 'basic_workflow/Outputs_Evaluation/BenchmarkDataset2025' #extended_workflow
root_old = 'basic_workflow/Outputs_Evaluation_OLD/BenchmarkDataset2025'#extended_workflow

# Loop through all subdirectories in the NEW folder
for subdir, dirs, files in os.walk(root_new):
    for file in files:
        if file.endswith('.xlsx'):
            relative_path = os.path.relpath(subdir, root_new)
            new_file_path = os.path.join(root_new, relative_path, file)
            old_file_path = os.path.join(root_old, relative_path, file)

            if os.path.exists(old_file_path):
                try:
                    print(f"\n📄 Comparing: {relative_path}/{file}")
                    
                    df_new = pd.read_excel(new_file_path)
                    df_old = pd.read_excel(old_file_path)

                    print(f"Rows - New: {len(df_new)} | Old: {len(df_old)}")

                    # Find common columns
                    common_cols = df_old.columns.intersection(df_new.columns)

                    # Reset index and compare only common columns
                    diff = df_old[common_cols].reset_index(drop=True).compare(
                        df_new[common_cols].reset_index(drop=True)
                    )

                    if diff.empty:
                        print("✅ No differences in common columns.")
                    else:
                        print("❗ Differences found in common columns:")
                        print(diff.head(5))  # Show top 5 differences

                except Exception as e:
                    print(f"❌ Failed to compare {file}: {e}")


📄 Comparing: finetuned_gemma2_9b_local/finetuned_gemma2_9b_local.xlsx
Rows - New: 2011 | Old: 2011
✅ No differences in common columns.

📄 Comparing: gemma2_9b/gemma2_9b.xlsx
Rows - New: 2012 | Old: 2012
✅ No differences in common columns.

📄 Comparing: finetuned_azzedde_llama3_1_8b/finetuned_azzedde_llama3_1_8b.xlsx
Rows - New: 2128 | Old: 2128
✅ No differences in common columns.

📄 Comparing: llama3_2_3b_local/llama3_2_3b_local.xlsx
Rows - New: 2128 | Old: 2128
✅ No differences in common columns.

📄 Comparing: llama3_1_8b/llama3_1_8b.xlsx
Rows - New: 2128 | Old: 2128
✅ No differences in common columns.

📄 Comparing: gpt_4o/gpt_4o.xlsx
Rows - New: 2128 | Old: 2128
✅ No differences in common columns.


In [32]:
df1 = pd.read_excel('basic_workflow/Outputs_Evaluation/BenchmarkDataset2025/gpt_4o/gpt_4o.xlsx')
df2 = pd.read_excel('extended_workflow/Outputs_Evaluation/BenchmarkDataset2025/gpt_4o/gpt_4o.xlsx')
# Find rows where 'score' is different
mask = df1['Original_KG_Valid'] != df2['Original_KG_Valid']

# Print rows with differing values
print("Rows where 'score' differs:")
print(df1[mask])
print(df2[mask])


Rows where 'score' differs:
Empty DataFrame
Columns: [question, schema, cypher, data_source, instance_id, database_reference_alias, url, database_name, username, password, Generated_Cypher, Output, Syntax Valid, Syntax Metadata, Schema Valid, Schema Metadata, Properties Valid, Properties Metadata, Pass@1 Output, Jaccard Output, BLEU, Rouge-L Precision, Rouge-L Recall, Rouge-L F1-score, Jaro Similarity, Jaccard Similarity, Original Syntax Valid, Original Syntax Metadata, Original Schema Valid, Original Schema Metadata, Original Properties Valid, Original Properties Metadata, Original_KG_Valid]
Index: []

[0 rows x 33 columns]
Empty DataFrame
Columns: [question, schema, cypher, data_source, instance_id, database_reference_alias, url, database_name, username, password, Generated_Cypher, answer, path, output, cypher_errors_history, cypher_errors, is_KG_Valid_correct, is_KG_Valid_correct_with_full_schema, Syntax Valid, Syntax Metadata, Schema Valid, Schema Metadata, Properties Valid, Proper

## Pass=None and Output Jaccard = None for all not KG valid queries 

#### Add to basic is_KG_Valid_correct column

In [64]:

for root_new in ['basic_workflow/Outputs_Evaluation/BenchmarkDataset2025']:#, 'extended_workflow/Outputs_Evaluation/BenchmarkDataset2025' 
    # Loop through all subdirectories in the NEW folder
    for subdir, dirs, files in os.walk(root_new):
        for file in files:
            if file.endswith('.xlsx'):
                relative_path = os.path.relpath(subdir, root_new)
                new_file_path = os.path.join(root_new, relative_path, file)
                print(f"\n📄 Reading file: {new_file_path}")
                df_new = pd.read_excel(new_file_path)
                kg_valid = (
                    (df_new['Schema Valid'] == 1) &
                    ((df_new['Properties Valid'] == 1) | (df_new['Properties Valid'].isna())) &
                    (df_new['Syntax Valid'] == True)
                )
                # Add the boolean result as a new column
                df_new['is_KG_Valid_correct'] = kg_valid

                # if 'generated_cypher' in df_new.columns:
                #     # Rename the column
                #     print('Renaming column generated_cypher to Generated_Cypher')
                #     df_new.rename(columns={'generated_cypher': 'Generated_Cypher'}, inplace=True)
                #     # Save to Excel (overwrite the same file)
                df_new.to_excel(new_file_path, index=False)

                # Save to Pickle
                pickle_file_path = new_file_path.replace('.xlsx', '.pkl')
                df_new.to_pickle(pickle_file_path)
                print('saved')

                
                # print(len(df_new.columns))


📄 Reading file: basic_workflow/Outputs_Evaluation/BenchmarkDataset2025/finetuned_gemma2_9b_local/finetuned_gemma2_9b_local.xlsx
saved

📄 Reading file: basic_workflow/Outputs_Evaluation/BenchmarkDataset2025/gemma2_9b/gemma2_9b.xlsx
saved

📄 Reading file: basic_workflow/Outputs_Evaluation/BenchmarkDataset2025/finetuned_azzedde_llama3_1_8b/finetuned_azzedde_llama3_1_8b.xlsx
saved

📄 Reading file: basic_workflow/Outputs_Evaluation/BenchmarkDataset2025/llama3_2_3b_local/llama3_2_3b_local.xlsx
saved

📄 Reading file: basic_workflow/Outputs_Evaluation/BenchmarkDataset2025/llama3_1_8b/llama3_1_8b.xlsx
saved

📄 Reading file: basic_workflow/Outputs_Evaluation/BenchmarkDataset2025/gpt_4o/gpt_4o.xlsx
saved


### The is_KG_Valid_correct = False should have None in Jaccard and Passs1 Output

In [67]:

for root_new in ['basic_workflow/Outputs_Evaluation/BenchmarkDataset2025', 'extended_workflow/Outputs_Evaluation/BenchmarkDataset2025' ]:
    # Loop through all subdirectories in the NEW folder
    for subdir, dirs, files in os.walk(root_new):
        for file in files:
            if file.endswith('.xlsx'):
                relative_path = os.path.relpath(subdir, root_new)
                new_file_path = os.path.join(root_new, relative_path, file)
                print(f"\n📄 Reading file: {new_file_path}")
                df_new = pd.read_excel(new_file_path)
                
                print(len(df_new[df_new['is_KG_Valid_correct'] == False]))
                df_new.loc[df_new['is_KG_Valid_correct'] == False, ['Pass@1 Output', 'Jaccard Output']] = None

               
                df_new.to_excel(new_file_path, index=False)

                # Save to Pickle
                pickle_file_path = new_file_path.replace('.xlsx', '.pkl')
                df_new.to_pickle(pickle_file_path)
                print('saved')

                
                # print(len(df_new.columns))


📄 Reading file: basic_workflow/Outputs_Evaluation/BenchmarkDataset2025/finetuned_gemma2_9b_local/finetuned_gemma2_9b_local.xlsx
457
saved

📄 Reading file: basic_workflow/Outputs_Evaluation/BenchmarkDataset2025/gemma2_9b/gemma2_9b.xlsx
891
saved

📄 Reading file: basic_workflow/Outputs_Evaluation/BenchmarkDataset2025/finetuned_azzedde_llama3_1_8b/finetuned_azzedde_llama3_1_8b.xlsx
1299
saved

📄 Reading file: basic_workflow/Outputs_Evaluation/BenchmarkDataset2025/llama3_2_3b_local/llama3_2_3b_local.xlsx
1242
saved

📄 Reading file: basic_workflow/Outputs_Evaluation/BenchmarkDataset2025/llama3_1_8b/llama3_1_8b.xlsx
1199
saved

📄 Reading file: basic_workflow/Outputs_Evaluation/BenchmarkDataset2025/gpt_4o/gpt_4o.xlsx
84
saved

📄 Reading file: extended_workflow/Outputs_Evaluation/BenchmarkDataset2025/finetuned_gemma2_9b_local/finetuned_gemma2_9b_local.xlsx
108
saved

📄 Reading file: extended_workflow/Outputs_Evaluation/BenchmarkDataset2025/llama3_2_3b/llama3_2_3b.xlsx
500
saved

📄 Reading fil